# Window Length Experiments
## Premise:
The original architecture was built for rock/pop music, which has many short, repeating passages. Classical music has longer passages and notes are often held for longer. 

Furthermore, classical music changes rhythms throughout a piece of music. The repetition still exists but over a more dynamic temporal context.

This notebook deals with trying to capture melodic patterns by increaing the temporal window. 

## Hypothesis: 
By expanding the window length and convolution dilation we can capture more of the long patterns in classical music

## Where to find results
Saves every epoch checkpoint plus final model and `eval.py` outputs to `/content/drive/MyDrive/Cadenza Challenge`.

## Running this Notebook
This notebook should be run on the Colab T4 GPU. Before running the notebook run `00_shared_drive_data_b.ipynb` to set up the Colab Server with the project data and libraries. This is different from `00_shared_drive_data.ipynb`, which sets up a different folder structure.

## Experiments
Window Length: 8s, 12s

Xblocks 8s: 10(default), 13,14,15
Xblocks 

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

print("GPU check:")
!nvidia-smi

AUDIO_ROOT = Path("/content/cadenza_data/cad2/task2/audio")
DRIVE_ROOT = Path("/content/drive/MyDrive/Cadenza Challenge")
REPO_ROOT = Path("/content/hearing-adaptive-music-processor")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

assert AUDIO_ROOT.exists(), f"Missing audio directory: {AUDIO_ROOT}"
print("Audio root verified:", AUDIO_ROOT)
print("Drive root ready:", DRIVE_ROOT)

# Adjust CONVTASNET_DIR to match the exact path of your ConvTasNet recipe folder.
# Note: train.py reads "local/conf.yml" relatively, so cd into CONVTASNET_DIR before running.
CONVTASNET_DIR = REPO_ROOT / "third_party/clarity/recipes/cad2/task2/ConvTasNet"
if CONVTASNET_DIR.exists():
    os.chdir(CONVTASNET_DIR)
print("Current working directory:", os.getcwd())

In [ ]:
# --- Experiment 1 Configuration ---
EXP_NAME = "exp1_segment_8s_X10"      # Name each ablation run clearly
SEGMENT_LENGTH = 8.0                  # Expanded window length in seconds (baseline: 3.0)
SAMPLES_PER_TRACK = 64                # Lower e.g. to 32 if longer segments slow epochs/I/O
BATCH_SIZE = 4
AGGREGATE = 2                         # Effective batch = BATCH_SIZE * AGGREGATE
EPOCHS = 200
NUM_WORKERS = 4
LR = 0.0001                           # Matches conf.yml default; adjust if needed

# Optional Conv-TasNet block overrides (defaults in conf.yml: X=10, R=4)
N_BLOCKS_X = 10
N_REPEATS_R = 4

METADATA_DIR = AUDIO_ROOT.parent / "metadata"  # /content/cadenza_data/cad2/task2/metadata

# Set to "latest" to auto-resume from newest checkpoint in Drive,
# a specific "/content/drive/MyDrive/Cadenza Challenge/.../*.ckpt" path, or None for fresh start.
RESUME_MODE = None#"latest"
EXPLICIT_CKPT_PATH = None

EXP_LOCAL_DIR = Path("exp") / EXP_NAME
DRIVE_EXP_DIR = DRIVE_ROOT / EXP_NAME
DRIVE_CKPT_DIR = DRIVE_EXP_DIR / "checkpoints"
EVAL_OUT_DIR = "eval_out"

EXP_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
assert METADATA_DIR.exists(), f"Missing metadata directory: {METADATA_DIR}"


def resolve_resume_checkpoint(ckpt_dir: Path, mode, explicit_path):
    if explicit_path:
        p = Path(explicit_path)
        assert p.exists(), f"Explicit checkpoint not found: {p}"
        return str(p)
    if mode == "latest":
        ckpts = sorted(ckpt_dir.rglob("*.ckpt"), key=lambda p: p.stat().st_mtime)
        if ckpts:
            return str(ckpts[-1])
    return None


ckpt_to_resume = resolve_resume_checkpoint(DRIVE_CKPT_DIR, RESUME_MODE, EXPLICIT_CKPT_PATH)
print("Local exp dir:", EXP_LOCAL_DIR.resolve())
print("Drive exp dir:", DRIVE_EXP_DIR)
print("Resuming from checkpoint:", ckpt_to_resume)

In [ ]:
import time
train_cmd = [
    "python", "train.py",
    "--exp_dir", str(EXP_LOCAL_DIR),
    "--drive_ckpt_dir", str(DRIVE_CKPT_DIR),
    "--data.root_path", str(AUDIO_ROOT),
    "--data.music_tracks_file", str(METADATA_DIR),
    "--data.segment_length", str(SEGMENT_LENGTH),
    "--data.samples_per_track", str(SAMPLES_PER_TRACK),
    "--convtasnet.X", str(N_BLOCKS_X),
    "--convtasnet.R", str(N_REPEATS_R),
    "--training.batch_size", str(BATCH_SIZE),
    "--training.aggregate", str(AGGREGATE),
    "--training.epochs", str(EPOCHS),
    "--training.num_workers", str(NUM_WORKERS),
    "--optim.lr", str(LR),
]

if ckpt_to_resume:
    train_cmd.extend(["--ckpt_path", ckpt_to_resume])

print("Running:", " ".join(train_cmd))

log_path = f"/content/drive/MyDrive/Cadenza Challenge/{EXP_NAME}gpu_mem_log.csv"
with open(log_path, "w") as gpu_log_file:
    gpu_proc = subprocess.Popen(
        [
            "nvidia-smi",
            "--query-gpu=timestamp,memory.used,memory.free,utilization.gpu",
            "--format=csv,noheader,nounits",
            "-lms", "1000",
        ],
        stdout=gpu_log_file,
    )
    try:
        subprocess.run(train_cmd, check=True)
    finally:
        gpu_proc.terminate()
        gpu_proc.wait()

In [ ]:
eval_cmd = [
    "python", "-u", "eval.py",
    "--exp_dir", str(EXP_LOCAL_DIR),
    "--out_dir", EVAL_OUT_DIR,
    "--use_gpu", "1" if os.path.exists("/dev/nvidia0") else "0",
    "--n_save_ex", "5",
]

print("Running:", " ".join(eval_cmd))
subprocess.run(eval_cmd, check=True)

# Copy final model artifacts & evaluation outputs to Google Drive for upstream enhancement
artifacts_to_copy = [
    EXP_LOCAL_DIR / "best_model.pth",
    EXP_LOCAL_DIR / "conf.yml",
    EXP_LOCAL_DIR / "best_k_models.json",
]

for artifact in artifacts_to_copy:
    if artifact.exists():
        shutil.copy2(artifact, DRIVE_EXP_DIR / artifact.name)
        print("Saved to Drive:", DRIVE_EXP_DIR / artifact.name)

local_eval_dir = EXP_LOCAL_DIR / EVAL_OUT_DIR
drive_eval_dir = DRIVE_EXP_DIR / EVAL_OUT_DIR
if local_eval_dir.exists():
    if drive_eval_dir.exists():
        shutil.rmtree(drive_eval_dir)
    shutil.copytree(local_eval_dir, drive_eval_dir)
    print("Saved eval results to Drive:", drive_eval_dir)

In [ ]:
import torch

saved_model_path = DRIVE_EXP_DIR / "best_model.pth"
results_txt_path = DRIVE_EXP_DIR / EVAL_OUT_DIR / "results.txt"

assert saved_model_path.exists(), f"Missing {saved_model_path}"
ckpt_obj = torch.load(saved_model_path, map_location="cpu")
print("best_model.pth keys:", list(ckpt_obj.keys()))

if results_txt_path.exists():
    print("\n--- Evaluation Summary (tail) ---")
    lines = results_txt_path.read_text().splitlines()
    print("\n".join(lines[-25:]))